# Ten real FeTA fetal brains on nine matched growth charts

This local meeting notebook uses FeTA 2.2 SVR images and automatic FetalSynthSeg predictions, not synthetic data. It fits only QC-passing cases labeled `Neurotypical`, then displays five neurotypical and five pathological examples. A reference flag is a research screen, not a diagnosis. FeTA data and derived images remain subject to FeTA access terms and are not committed.

In [ ]:
from pathlib import Path
import json
import sys
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
if not (ROOT/'src/fetal_brain_growth').is_dir():
    raise FileNotFoundError(f'Run this notebook from the fetal-brain-growth repository; cwd={ROOT}')
sys.path.insert(0, str(ROOT/'src'))
import pandas as pd
from IPython.display import Image, display
from fetal_brain_growth.charts import save_growth_chart
from fetal_brain_growth.feta_gallery import build_feta_gallery
from fetal_brain_growth.feta_reference import resolve_feta_root
from fetal_brain_growth.labels import REFERENCE_GROUPS
from fetal_brain_growth.references import build_table_curves, score_against_curves

In [ ]:
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
FETA_ROOT = resolve_feta_root()  # or resolve_feta_root('/path/to/feta_2.2')
OUT = ROOT/'meeting_outputs/feta_10_cases_matched'
CHECKPOINT = ROOT/'models/KISPI-all_fss.ckpt'
paths = build_feta_gallery(
    FETA_ROOT, OUT, reference='feta-neurotypical', feta_degree=2,
    checkpoint=CHECKPOINT, device_name='auto',
)
print('FeTA root:', FETA_ROOT)
print('Meeting outputs:', OUT)

## Cohort selection and independent labels

The gallery deliberately includes both FeTA phenotype groups. Phenotype is supplied by the dataset; the volumetric screen is calculated here. A pathological phenotype may have all volumes within range, and a reference flag alone does not establish pathology.

In [ ]:
summary = pd.read_csv(paths['summary'])
display(summary[['subject_id','gestational_age_weeks','feta_phenotype','volume_screen','reference_result_detail']])

## Automatic segmentations in standard orientation

In [ ]:
display(Image(filename=str(paths['overview']), width=1500))

## Nine exact-label volume charts

The panels show total brain, intracranial volume, external CSF, cortical gray matter, white matter, ventricles, cerebellum, deep gray matter, and brainstem with P3/P10/P25/P50/P75/P90/P97 bands. Red points fall outside P3–P97.

In [ ]:
display(Image(filename=str(paths['growth_chart']), width=1500))

## The same ten cases on Ren 2022 reconstructed quantiles

This second chart uses `mean(GA) + Φ⁻¹(q) × SD(GA)` reconstructed from the Ren weekly summary table. It is included to show how conclusions depend on the reference population and anatomical definitions. Green/red points are the four definition-aligned screens (total brain, intracranial volume, external CSF, and cerebellum); orange points are comparison-only. Do not interpret differences between the two charts as biological change in an individual fetus.

In [ ]:
matched_scores = pd.read_csv(paths['scores'])
base_volumes = matched_scores[['subject_id','gestational_age_weeks','region','volume_ml']].copy()
subcortical = (
    base_volumes[base_volumes.region.isin(['white_matter','deep_gray_matter'])]
    .groupby(['subject_id','gestational_age_weeks'], as_index=False).volume_ml.sum()
)
subcortical['region'] = 'subcortical_brain_tissue'
ren_volumes = pd.concat([
    base_volumes[base_volumes.region.isin(REFERENCE_GROUPS) & (base_volumes.region != 'subcortical_brain_tissue')],
    subcortical,
], ignore_index=True)
ren_curves, ren_metadata = build_table_curves(method='interpolate')
ren_scores = score_against_curves(ren_volumes, ren_curves, definition_guard=True)
ren_chart = save_growth_chart(
    ren_curves, OUT/'ten_case_growth_chart_ren2022.png', observations=ren_scores,
    regions=tuple(REFERENCE_GROUPS),
    title='The same ten FeTA cases on Ren 2022 reconstructed quantiles',
    subtitle='Normal approximation from weekly mean/SD • green/red = definition-aligned; orange = comparison-only',
    dpi=240,
)
display(Image(filename=str(ren_chart), width=1500))
display(ren_scores[['subject_id','gestational_age_weeks','region','volume_ml','percentile_display','status']])

## Flagged measurements and per-case report

In [ ]:
scores = pd.read_csv(paths['scores'])
flagged = scores[scores.status.isin(['low_reference_flag','high_reference_flag'])]
display(flagged[['subject_id','gestational_age_weeks','region','volume_ml','percentile_display','status']])
display(Image(filename=str(OUT/'case_cards/sub-050_case_report.png'), width=1500))

## How the reference was generated

This is a **protocol-matched teaching reference**, not a validated clinical norm and not the Ren summary-data reconstruction shown in the second chart. It is built as follows:

1. Select FeTA rows whose `Pathology` field is exactly `Neurotypical` (case-insensitive).
2. Run the frozen FetalSynthSeg model and measure its seven predicted-label volumes plus total brain and intracranial volume using `voxel count × abs(det(affine[:3,:3])) / 1000` mL.
3. Exclude technical segmentation-QC failures, but remove no cases based on their measured volume. The exact included count and age range are recorded in the generated metadata.
4. For each region fit `log(V) = β₀ + β₁(GA − mean(GA)) + β₂(GA − mean(GA))²`.
5. Estimate one constant residual SD `s` in log-volume space and calculate `Q_q(GA) = exp(fitted_log_volume(GA) + Φ⁻¹(q) × s)` for P3/P10/P25/P50/P75/P90/P97.

These are estimated population intervals under a log-Normal residual assumption, not confidence intervals for the fitted median. Case percentiles are linearly interpolated between the seven curves within P3–P97; values outside are shown as `P3 or lower` or `P97 or higher` and receive a research flag. Cubic fitting is available with `fbg feta-reference --degree 3`, but is only a sensitivity analysis for this small cohort. Because some displayed normal cases also helped fit the curves, their positions are in-sample.

In [ ]:
metadata = json.loads(Path(paths['metadata']).read_text())
print({key: metadata[key] for key in [
    'subjects','age_min_weeks','age_max_weeks','degree','quantiles','segmentation_qc_excluded_cases']})
diagnostics = pd.DataFrame(metadata['diagnostics']).T.reset_index(names='region')
display(diagnostics[['region','log_volume_r_squared','leave_one_out_rmse_log_volume','log_residual_sd']].style.format({
    'log_volume_r_squared': '{:.3f}',
    'leave_one_out_rmse_log_volume': '{:.3f}',
    'log_residual_sd': '{:.3f}',
}))

## Interpretation limits

- The FeTA control set is small, cross-sectional, and partly reused as displayed normal examples.
- Do not extrapolate outside 22.7–34.8 weeks.
- Ventricular, deep-gray, and brainstem fits are especially uncertain; inspect diagnostics.
- Review the complete 3-D image and segmentation, gestational-age uncertainty, morphology, and clinical context with a fetal neuroradiologist.
- A larger independent cohort processed with the same frozen pipeline is required for clinical validation.